<a href="https://colab.research.google.com/github/viviantram03/labb-1/blob/main/labaml3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math

In [16]:
class GRUTagger(nn.Module):
  def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
    super(GRUTagger, self).__init__()
    self.hidden_dim = hidden_dim
    self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
    self.gru = nn.GRU(embedding_dim, hidden_dim)
    self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

  def forward(self, sentence):
    embeds = self.word_embeddings(sentence)
    gru_out, hidden = self.gru(embeds.view(len(sentence), 1, -1))
    tag_space = self.hidden2tag(gru_out.view(len(sentence), -1))
    tag_scores = F.log_softmax(tag_space, dim=1)
    return tag_scores

def train_and_compare(models, training_data, word_to_ix, tag_to_ix, epochs=100):
  loss_function = nn.NLLLoss()
  results = {}

  for name, model in models.items():
    optimizer = optim.SGD(model.parameters(), lr=0.1)
    last_loss = 0
    for epoch in range(epochs):
      total_loss = 0
      for sentence, tags in training_data:
        model.zero_grad()
        sentence_in = prepare_sequence(sentence, word_to_ix)
        targets = prepare_sequence(tags, tag_to_ix)
        tag_scores = model(sentence_in)
        loss = loss_function(tag_scores, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
      last_loss = total_loss / len(training_data)

    perplexity = math.exp(last_loss)
    results[name] = {"Loss": last_loss, "Perplexity": perplexity}
    print(f"{name} - Loss: {last_loss:.4f}, Perplexity: {perplexity:.4f}")
  return results


In [17]:
class SkipGram(nn.Module):
  def __init__(self, vocab_size, embedding_dim):
    super(SkipGram, self).__init__()
    self.embeddings = nn.Embedding(vocab_size, embedding_dim)
    self.linear = nn.Linear(embedding_dim, vocab_size)

  def forward(self, inputs):
    embeds = self.embeddings(inputs)
    out = self.linear(embeds)
    return F.log_softmax(out, dim=1)

def create_skipgram_data(raw_text, window=2):
  data = []
  for i in range(window, len(raw_text) - window):
    target = raw_text[i]
    context = [raw_text[i-j] for j in range(1, window+1)] + [raw_text[i+j] for j in range(1, window+1)]
    for word in context:
      data.append((target, word))
  return data

def train_skipgram(model, data, word_to_ix, epochs=50):
  optimizer = optim.SGD(model.parameters(), lr=0.001)
  loss_fn = nn.NLLLoss()
  for epoch in range(epochs):
    total_loss = 0
    for target, context in data:
      target_idx = torch.tensor([word_to_ix[target]], dtype=torch.long)
      context_idx = torch.tensor([word_to_ix[context]], dtype=torch.long)
      model.zero_grad()
      log_probs = model(context_idx)
      loss = loss_fn(log_probs, target_idx)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()


In [18]:
class NameClassifierGRU(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super(NameClassifierGRU, self).__init__()
    self.hidden_size = hidden_size
    self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
    self.fc = nn.Linear(hidden_size, output_size)
    self.softmax = nn.LogSoftmax(dim=1)

  def forward(self, line_tensor):
    line_tensor = line_tensor.transpose(0,1)
    _, hidden = self.gru(line_tensor)
    output = self.fc(hidden.squeeze(0))
    return self.softmax(output)